# MPC Sweep — Shard Runner (CPU-only)

Runs a subset of the 35 MPC cells (7 presets x 5 seeds).
Open 5 copies of this notebook in parallel, each with a different SHARD_ID.

**IMPORTANT**: Runtime -> Change runtime type -> **CPU** (not GPU!).
MPC's `mpc_planner.batch_predict` has no device propagation; the production
code runs CPU-only. Decision Gate confirmed Colab T4 is 0.57x slower than
local 12-thread CPU for MPC, so CPU is the only sensible choice.

**Parallelism**: 5 shards x 7 cells each = 35 total. Each shard ~15-20h wall.
Use `--limit-time 72000` (20h) to stay within Colab's 24h session limit.

In [ ]:
# === CONFIG: change SHARD_ID for each parallel session ===
SHARD_ID = 0       # 0, 1, 2, 3, or 4
TOTAL_SHARDS = 5
LIMIT_TIME_S = 72000  # 20h safety margin within 24h session
print(f'Shard {SHARD_ID}/{TOTAL_SHARDS}, limit={LIMIT_TIME_S}s')

In [ ]:
# Cell 1: Mount Drive + unzip bundle
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/arcgis-farmland-mpc'
BUNDLE = f'{DRIVE_ROOT}/mpc_sweep_bundle.zip'
assert os.path.exists(BUNDLE), f'Upload mpc_sweep_bundle.zip to {DRIVE_ROOT}'
!unzip -q -o $BUNDLE -d /content
!ls /content/src

In [ ]:
# Cell 2: Install deps (no GPU libs needed)
!pip install -q torch geopandas libpysal opensimplex rasterio pyyaml shapely

In [ ]:
# Cell 3: Env vars
import os
os.environ['TEST_SRC_ROOT'] = '/content/src/test'
os.environ['ADK_SRC_ROOT']  = '/content/src/adk'
import torch
torch.set_num_threads(os.cpu_count() or 4)
print(f'torch threads: {torch.get_num_threads()}')
print(f'cpu count: {os.cpu_count()}')

In [ ]:
# Cell 4: Generate all 35 datasets (shared across shards, ~30 min)
%cd /content/src/benchmark
import pathlib
presets = ['bishan_clone','neijiang_clone','plain_small_cons','plain_large_cons',
           'plain_medium_frag','mixed_medium_frag','hilly_small_cons']
for p in presets:
    for s in range(5):
        out = f'data_dev/{p}_seed{s}'
        if pathlib.Path(f'{out}/manifest.json').exists():
            print(f'skip {out}')
            continue
        !python -m generator.generate --preset presets/{p}.yaml --seed {s} --out {out}
print('Done: datasets generated')

In [ ]:
# Cell 5: Run MPC sweep for this shard
%cd /content/src/benchmark
!python -m sweep.runner \
  --manifest sweep_manifest.csv \
  --state sweep_state_mpc_shard{SHARD_ID}.json \
  --data-root data_dev \
  --results-root results \
  --only-method MPC \
  --shard {SHARD_ID}/{TOTAL_SHARDS} \
  --resume \
  --limit-time {LIMIT_TIME_S}

In [ ]:
# Cell 6: Package results + state for download
%cd /content/src/benchmark
import shutil
out_zip = f'/content/mpc_results_shard{SHARD_ID}.zip'
!cd /content/src/benchmark && zip -r {out_zip} results/ sweep_state_mpc_shard{SHARD_ID}.json
drive_out = f'{DRIVE_ROOT}/mpc_results_shard{SHARD_ID}.zip'
shutil.copy(out_zip, drive_out)
print(f'Results saved to {drive_out}')
print(f'Also at {out_zip} for direct download')